In [0]:
%sql
-- widget for end date

-- ============================================================
-- 0) Helper: drop/replace convenience (optional)
-- ============================================================
-- DROP VIEW IF EXISTS payer_rollup_dim;
-- DROP VIEW IF EXISTS total_lives;
-- DROP VIEW IF EXISTS payer_master_patient_level;
-- DROP VIEW IF EXISTS all_patient_claims;
-- DROP VIEW IF EXISTS all_patient_claims_expanded;
-- DROP VIEW IF EXISTS elaprase_provider_universe;
-- DROP TABLE IF EXISTS com_edp_prd.cmpa_insights_internal_schema.payer360_master;
-- DROP TABLE IF EXISTS cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL;
-- DROP TABLE IF EXISTS cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL;

-- ============================================================
-- 1) PAYER ROLLUP DIM (single source of truth)
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer_rollup_dim AS
SELECT DISTINCT
    PAYER_ID,
    PAYER_NAME,

    CASE
        WHEN UPPER(PAYER_NAME) RLIKE 'UNITED|UHC|OPTUM' THEN 'UHC/Optum'
        WHEN UPPER(PAYER_NAME) RLIKE 'CIGNA|ESI|EVERNORTH' THEN 'Cigna/ESI'
        WHEN UPPER(PAYER_NAME) RLIKE 'AETNA|CVS' THEN 'Aetna/CVS'
        WHEN UPPER(PAYER_NAME) RLIKE 'ANTHEM|ELEVANCE|EMPIRE' THEN 'Elevance/Carelon'
        ELSE PAYER_NAME
    END AS payer_display_name,

    CASE
        WHEN UPPER(PAYER_NAME) RLIKE 'UNITED|UHC|OPTUM' THEN 'ROLLUP_UHC_OPTUM'
        WHEN UPPER(PAYER_NAME) RLIKE 'CIGNA|ESI|EVERNORTH' THEN 'ROLLUP_CIGNA_ESI'
        WHEN UPPER(PAYER_NAME) RLIKE 'AETNA|CVS' THEN 'ROLLUP_AETNA_CVS'
        WHEN UPPER(PAYER_NAME) RLIKE 'ANTHEM|ELEVANCE|EMPIRE' THEN 'ROLLUP_ELEVANCE'
        ELSE CAST(PAYER_ID AS STRING)
    END AS payer_display_id

FROM com_edp_prd.com_raw.kom_plans
WHERE PAYER_ID IS NOT NULL;

-- ============================================================
-- 2) TOTAL_LIVES (use rolling payer_display_id)
-- ============================================================
CREATE OR REPLACE TEMP VIEW total_lives AS

WITH all_claims AS (
  SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    COALESCE(KH_PLAN_ID) AS plan_id,
    MEDICAL_EVENT_ID AS claim_id,
    'MEDICAL' AS src
  FROM com_edp_prd.com_raw.kom_medical_events

  UNION ALL

  SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
    PHARMACY_EVENT_ID AS claim_id,
    'PHARMACY' AS src
  FROM com_edp_prd.com_raw.kom_pharmacy_events
),

tagged AS (
  SELECT
    c.PATIENT_ID,
    c.NPI,
    c.plan_id,
    c.claim_id,
    COALESCE(pr.payer_display_id, 'Unknown') AS payer_display_id,
    COALESCE(pr.payer_display_name, 'Unknown') AS payer_display_name,
    COALESCE(CAST(z.territory_id AS STRING), 'Unknown') AS territory_id,
    COALESCE(z.territory_name, 'Unknown') AS territory
  FROM all_claims c
  LEFT JOIN com_edp_prd.com_raw.kom_plans pl
    ON c.plan_id = pl.KH_PLAN_ID
  LEFT JOIN payer_rollup_dim pr
    ON pl.PAYER_ID = pr.PAYER_ID
  LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON TRY_CAST(
         (SELECT PROVIDER_ZIP FROM com_edp_prd.com_raw.kom_providers p WHERE p.NPI = c.NPI AND p.PROVIDER_TYPE = 'INDIVIDUAL' LIMIT 1)
       AS STRING) = TRY_CAST(z.zipcode AS STRING)
),

agg AS (
  SELECT
    COALESCE(territory_id, 'ALL Territories')         AS territory_id,
    COALESCE(territory, 'All Territories')            AS territory,
    COALESCE(payer_display_id, 'ALL Payers')          AS payer_display_id,
    COALESCE(payer_display_name, 'All Payers')        AS payer_display_name,
    COUNT(DISTINCT patient_id)                        AS total_lives,
    CASE
      WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=0 THEN 'TERRITORY_PAYER'
      WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=0 THEN 'PAYER_ALL_TERRITORY'
      WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=1 THEN 'TERRITORY_ALL_PAYER'
      WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=1 THEN 'NATIONAL'
    END AS rollup_level
  FROM tagged
  GROUP BY GROUPING SETS (
    (territory_id, territory, payer_display_id, payer_display_name),
    (payer_display_id, payer_display_name),
    (territory_id, territory),
    ()
  )
)

SELECT * FROM agg
ORDER BY rollup_level, total_lives DESC;


-- ============================================================
-- 3) Eligible patients / claim pulls (your existing logic)
-- (Keep your original definitions for all_dx_claims, all_tx_claims,
--  e761_patients_2dx, specified_patients, incremental_patients, eligible_patients,
--  all_patient_claims). I will assume they exist as in your notebook.
-- ============================================================
-- (Skipping re-definitions here to avoid duplication — use your existing views:
--  all_dx_claims, all_tx_claims, all_tx_claims, e761_patients_2dx, specified_patients,
--  e763_patients_2dx, elaprase_tx, incremental_patients, eligible_patients,
--  all_patient_claims)
-- If you want me to include them verbatim, say so and I will paste them.

-- ============================================================
-- 4) PAYER MASTER PATIENT LEVEL (keep raw payer as raw_payer_* and include rollup)
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer_master_patient_level AS

WITH -- eligible_patient_universe, patient_claim_counts, primary_hcp, latest_plan_per_patient
-- (use your existing logic for those, assumed present in the notebook)

eligible_patient_universe AS (
  SELECT DISTINCT patient_id FROM eligible_patients
),

patient_claim_counts AS (
  SELECT patient_id, COUNT(DISTINCT claim_id) AS claims_count
  FROM all_patient_claims
  GROUP BY patient_id
),

patients_with_primary_hcp AS (
  SELECT a.patient_id, a.claims_count, ph.hcp_npi, ph.hcp_name, ph.hcp_specialty, ph.hcp_zip, ph.hco_npi, ph.hco_name,
         ph.territory_id, ph.territory, ph.region_id, ph.region
  FROM eligible_patient_universe a
  LEFT JOIN (
    SELECT * FROM primary_hcp
  ) ph
  ON a.patient_id = ph.patient_id
),

latest_plan_per_patient AS (
  SELECT patient_id, plan_id
  FROM (
    SELECT patient_id, plan_id,
           ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY fill_date DESC, npi ASC) AS rn
    FROM all_patient_claims
    WHERE plan_id IS NOT NULL
  ) t
  WHERE rn = 1
),

patients_with_latest_plan AS (
  SELECT p.*, l.plan_id
  FROM patients_with_primary_hcp p
  LEFT JOIN latest_plan_per_patient l ON p.patient_id = l.patient_id
),

patients_with_payer_attributes AS (
  SELECT DISTINCT
    a.patient_id,
    COALESCE(a.claims_count,0) AS claims_count,
    a.hcp_npi,
    a.hcp_name,
    a.hcp_specialty,
    a.hcp_zip,
    a.hco_npi,
    a.hco_name,
    COALESCE(CAST(a.territory_id AS STRING),'Unknown') AS territory_id,
    COALESCE(a.territory,'Unknown') AS territory,
    a.region_id,
    COALESCE(a.region,'Unknown') AS region,

    a.plan_id,

    -- raw payer fields from plan table (keep them but rename to avoid duplicate naming)
    COALESCE(pl.PAYER_ID, 'Unknown')         AS raw_payer_id,
    COALESCE(pl.PAYER_NAME, 'Unknown')       AS raw_payer_name,
    COALESCE(pl.PARENT_ID, 'Unknown')        AS parent_id,
    COALESCE(pl.PARENT_NAME, 'Unknown')      AS parent_name,
    COALESCE(pl.INSURANCE_SEGMENT, 'Unknown') AS insurance_segment,
    COALESCE(pl.INSURANCE_GROUP, 'Unknown')   AS insurance_group

  FROM patients_with_latest_plan a
  LEFT JOIN com_edp_prd.com_raw.kom_plans pl
    ON a.plan_id = pl.KH_PLAN_ID
)

SELECT
  patient_id,
  claims_count,
  hcp_npi,
  hcp_name,
  hcp_specialty,
  hcp_zip,
  hco_npi,
  hco_name,
  territory_id,
  territory,
  region_id,
  region,
  plan_id,
  raw_payer_id,
  raw_payer_name,
  parent_id,
  parent_name,
  insurance_segment,
  insurance_group,

  -- Add rolled up payer attrs by joining to the canonical dimension
  COALESCE(pr.payer_display_id, 'Unknown')   AS payer_display_id,
  COALESCE(pr.payer_display_name, 'Unknown') AS payer_display_name

FROM patients_with_payer_attributes p
LEFT JOIN payer_rollup_dim pr
  ON p.raw_payer_id = pr.PAYER_ID;


-- ============================================================
-- 5) ELAPR A SE Provider Universe (provider-level view)
--    -> use payer_display_id (no collisions)
-- ============================================================
CREATE OR REPLACE TEMP VIEW elaprase_provider_universe AS

WITH raw_provider_claims AS (

    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
        BILLING_NPI AS HCO_NPI,
        KH_PLAN_ID AS plan_id
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (
            DIAGNOSIS_CODES LIKE '%E761%'
         OR DIAGNOSIS_CODES LIKE '%E763%'
         OR NDC11 IN ('54092070001','540920700')
         OR PROCEDURE_CODE IN (
             '99601','99602','96365','96366','J1743',
             'S9357','S9379','38206','38230','38232',
             '38240','38241','38242','38243','38250'
         )
    )
    AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION ALL

    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS HCP_NPI,
        PHARMACY_NPI AS HCO_NPI,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE (
            DIAGNOSIS_CODE IN ('E761','E763')
         OR NDC11 IN ('54092070001','540920700')
    )
    AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
    AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)

SELECT DISTINCT
    rpc.PATIENT_ID,
    rpc.HCP_NPI,
    rpc.HCO_NPI,
    COALESCE(pm.payer_display_id,'Unknown')   AS payer_display_id,
    COALESCE(pm.payer_display_name,'Unknown') AS payer_display_name,
    pm.parent_id,
    pm.parent_name,
    COALESCE(CAST(pm.territory_id AS STRING),'Unknown') AS territory_id,
    COALESCE(pm.territory,'Unknown') AS territory,
    pm.region_id,
    pm.region
FROM raw_provider_claims rpc
LEFT JOIN payer_master_patient_level pm
  ON rpc.PATIENT_ID = pm.patient_id;

-- ============================================================
-- 6) ENRICHED CLAIMS (force rolled-up payer into claim-level)
-- ============================================================
CREATE OR REPLACE TEMP VIEW all_patient_claims_expanded AS

SELECT
    c.patient_id,
    c.claim_id,
    c.fill_date,
    -- primary HCP attribution (from primary_hcp)
    p.hcp_npi,
    p.hcp_name,
    p.hcp_specialty,
    CASE WHEN p.hco_npi IS NULL THEN 'Unknown HCO' ELSE p.hco_npi END AS hco_npi,
    CASE WHEN p.hco_name IS NULL THEN 'Unknown HCO' ELSE p.hco_name END AS hco_name,
    p.territory_id,
    p.territory AS territory_name,
    p.region_id,
    p.region AS region_name,

    -- Use canonical rollup columns everywhere downstream
    COALESCE(p.payer_display_id, 'Unknown')   AS payer_display_id,
    COALESCE(p.payer_display_name, 'Unknown') AS payer_display_name,

    -- keep raw payer if needed (renamed)
    p.raw_payer_id,
    p.raw_payer_name,
    p.parent_id,
    p.parent_name

FROM all_patient_claims c
INNER JOIN payer_master_patient_level p
  ON c.patient_id = p.patient_id;

-- ============================================================
-- 7) ROLLUP METRICS (payer360 core) using payer_display_id
-- ============================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer360_master AS

WITH new_patient_flags AS (
  SELECT patient_id, MIN(fill_date) AS FIRST_EVENT_DATE,
         CASE WHEN MIN(fill_date) >= DATEADD(month,-1,DATE('${end_date}')) THEN 1 ELSE 0 END AS NEW_PATIENT_R1M,
         CASE WHEN MIN(fill_date) >= DATEADD(month,-3,DATE('${end_date}')) THEN 1 ELSE 0 END AS NEW_PATIENT_R3M
  FROM all_patient_claims
  GROUP BY patient_id
),

patient_claim_metrics AS (
  SELECT p.patient_id,
         COUNT(DISTINCT a.claim_id) AS TOTAL_CLAIMS,
         COUNT(DISTINCT ph.PHARMACY_EVENT_ID) AS PHARMACY_TOTAL_CLAIMS,
         COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='PAID' THEN ph.PHARMACY_EVENT_ID END) AS APPROVED_FILLS,
         COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='REJECTED' THEN ph.PHARMACY_EVENT_ID END) AS REJECTED_FILLS,
         COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='REVERSED' THEN ph.PHARMACY_EVENT_ID END) AS REVERSED_FILLS
  FROM payer_master_patient_level p
  LEFT JOIN all_patient_claims a ON p.patient_id = a.patient_id
  LEFT JOIN com_edp_prd.com_raw.kom_pharmacy_events ph
    ON p.patient_id = ph.PATIENT_ID
   AND ph.FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
   AND (ph.DIAGNOSIS_CODE IN ('E761','E763') OR ph.NDC11 IN ('54092070001','540920700'))
  GROUP BY p.patient_id
),

patient_age AS (
  SELECT p.patient_id, YEAR(CURRENT_DATE) - YEAR(d.PATIENT_YOB) AS CURRENT_AGE
  FROM payer_master_patient_level p
  LEFT JOIN (
    SELECT PATIENT_ID, MAX(PATIENT_YOB) AS PATIENT_YOB FROM com_edp_prd.com_raw.kom_patient_demographics GROUP BY PATIENT_ID
  ) d ON p.patient_id = d.PATIENT_ID
),

payer_group_map AS (
  SELECT p.*,
         pr.payer_display_id,
         pr.payer_display_name,
         CASE WHEN pr.payer_display_name IN ('UHC/Optum','Cigna/ESI','Aetna/CVS','Elevance/Carelon') THEN pr.payer_display_name ELSE p.raw_payer_name END AS payer_group
  FROM payer_master_patient_level p
  LEFT JOIN payer_rollup_dim pr ON p.raw_payer_id = pr.PAYER_ID
),

base_enriched AS (
  SELECT p.*,
         COALESCE(i.PIE_COMPLETED,'NO') AS PIE_COMPLETED,
         COALESCE(i.ACCOUNT_DIRECTOR,'-') AS ACCOUNT_DIRECTOR,
         COALESCE(n.NEW_PATIENT_R1M,0) AS NEW_PATIENT_R1M,
         COALESCE(n.NEW_PATIENT_R3M,0) AS NEW_PATIENT_R3M,
         COALESCE(c.TOTAL_CLAIMS,0)          AS TOTAL_CLAIMS,
         COALESCE(c.PHARMACY_TOTAL_CLAIMS,0) AS PHARMACY_TOTAL_CLAIMS,
         COALESCE(c.APPROVED_FILLS,0)        AS APPROVED_FILLS,
         COALESCE(c.REJECTED_FILLS,0)        AS REJECTED_FILLS,
         COALESCE(c.REVERSED_FILLS,0)        AS REVERSED_FILLS,
         a.CURRENT_AGE,
         CASE WHEN a.CURRENT_AGE < 5 THEN 1 ELSE 0 END AS AGE_LT_5_YRS,
         CASE WHEN a.CURRENT_AGE BETWEEN 5 AND 10 THEN 1 ELSE 0 END AS AGE_5_TO_10_YRS,
         CASE WHEN a.CURRENT_AGE BETWEEN 11 AND 18 THEN 1 ELSE 0 END AS AGE_11_TO_18_YRS,
         CASE WHEN a.CURRENT_AGE > 18 THEN 1 ELSE 0 END AS AGE_GT_18_YRS
  FROM payer_group_map p
  LEFT JOIN new_patient_flags n ON p.patient_id = n.patient_id
  LEFT JOIN patient_claim_metrics c ON p.patient_id = c.patient_id
  LEFT JOIN patient_age a ON p.patient_id = a.patient_id
  LEFT JOIN (
      SELECT PAYER_ACCOUNT_NAME, MAX(PIE_COMPLETED) AS PIE_COMPLETED, MAX(ACCOUNT_DIRECTOR) AS ACCOUNT_DIRECTOR
      FROM com_edp_prd.cmpa_insights_internal_schema.payer_info
      GROUP BY PAYER_ACCOUNT_NAME
  ) i ON p.PAYER_ACCOUNT_NAME_STD = i.PAYER_ACCOUNT_NAME
)

, rollup_metrics AS (
  SELECT
    COALESCE(CAST(territory_id AS STRING), 'ALL Territories') AS territory_id,
    COALESCE(territory, 'All Territories')                    AS territory_name,
    COALESCE(CAST(payer_display_id AS STRING), 'ALL Payers')   AS payer_display_id,
    COALESCE(payer_display_name, 'All Payers')                 AS payer_display_name,
    COALESCE(payer_group,'All Payers')                         AS payer_group,
    COALESCE(CAST(parent_id AS STRING), 'ALL Parents')         AS parent_id,
    COALESCE(parent_name, 'All Parents')                       AS parent_name,

    COUNT(DISTINCT patient_id) AS total_elaprase_patients,

    COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
    COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
    COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,
    COUNT(DISTINCT CASE WHEN insurance_group IS NULL OR insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL') THEN patient_id END) AS other_patients,

    SUM(NEW_PATIENT_R1M) AS NEW_ELAPRASE_PATIENTS_R1M,
    SUM(NEW_PATIENT_R3M) AS NEW_ELAPRASE_PATIENTS_R3M,

    COUNT(DISTINCT CASE WHEN AGE_LT_5_YRS = 1 THEN patient_id END)     AS AGE_LT_5_YRS,
    COUNT(DISTINCT CASE WHEN AGE_5_TO_10_YRS = 1 THEN patient_id END)  AS AGE_5_TO_10_YRS,
    COUNT(DISTINCT CASE WHEN AGE_11_TO_18_YRS = 1 THEN patient_id END) AS AGE_11_TO_18_YRS,
    COUNT(DISTINCT CASE WHEN AGE_GT_18_YRS = 1 THEN patient_id END)    AS AGE_GT_18_YRS,

    COUNT(DISTINCT hcp_npi) AS TOTAL_PRIMARY_HCPS,
    COUNT(DISTINCT hco_npi) AS TOTAL_PRIMARY_HCOS,

    SUM(TOTAL_CLAIMS)          AS TOTAL_CLAIMS,
    SUM(PHARMACY_TOTAL_CLAIMS) AS PHARMACY_TOTAL_CLAIMS,
    SUM(APPROVED_FILLS)        AS APPROVED_FILLS,
    SUM(REJECTED_FILLS)        AS REJECTED_FILLS,
    SUM(REVERSED_FILLS)        AS REVERSED_FILLS,

    CASE WHEN SUM(PHARMACY_TOTAL_CLAIMS)=0 THEN 0 ELSE ROUND(100.0*SUM(APPROVED_FILLS)/SUM(PHARMACY_TOTAL_CLAIMS),2) END AS ELAPRASE_APPROVAL_RATE,
    CASE WHEN SUM(PHARMACY_TOTAL_CLAIMS)=0 THEN 0 ELSE ROUND(100.0*SUM(REJECTED_FILLS)/SUM(PHARMACY_TOTAL_CLAIMS),2) END AS ELAPRASE_REJECTION_RATE,
    CASE WHEN SUM(PHARMACY_TOTAL_CLAIMS)=0 THEN 0 ELSE ROUND(100.0*SUM(REVERSED_FILLS)/SUM(PHARMACY_TOTAL_CLAIMS),2) END AS ELAPRASE_REVERSED_RATE,

    CASE WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=0 THEN MAX(PIE_COMPLETED)
         WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=0 THEN MAX(PIE_COMPLETED)
         ELSE 'ALL Payers' END AS PIE_COMPLETED,

    CASE WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=0 THEN MAX(ACCOUNT_DIRECTOR)
         WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=0 THEN MAX(ACCOUNT_DIRECTOR)
         ELSE 'ALL Payers' END AS ACCOUNT_DIRECTOR,

    CASE
      WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=0 THEN 'TERRITORY_PAYER'
      WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=0 THEN 'PAYER_ALL_TERRITORY'
      WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=1 THEN 'TERRITORY_ALL_PAYER'
      WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=1 THEN 'NATIONAL'
    END AS rollup_level

  FROM base_enriched
  GROUP BY GROUPING SETS (
    (territory_id, territory, payer_display_id, payer_display_name, payer_group, parent_id, parent_name),
    (payer_display_id, payer_display_name, payer_group, parent_id, parent_name),
    (territory_id, territory),
    ()
  )
)

, provider_universe_rollup AS (
  SELECT
    COALESCE(CAST(territory_id AS STRING),'ALL Territories') AS territory_id,
    COALESCE(territory,'All Territories') AS territory_name,
    COALESCE(CAST(payer_display_id AS STRING),'ALL Payers') AS payer_display_id,
    COALESCE(payer_display_name,'All Payers') AS payer_display_name,
    COUNT(DISTINCT hcp_npi) AS TOTAL_HCPS,
    COUNT(DISTINCT hco_npi) AS TOTAL_HCOS,
    CASE
      WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=0 THEN 'TERRITORY_PAYER'
      WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=0 THEN 'PAYER_ALL_TERRITORY'
      WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=1 THEN 'TERRITORY_ALL_PAYER'
      WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=1 THEN 'NATIONAL'
    END AS rollup_level
  FROM elaprase_provider_universe
  GROUP BY GROUPING SETS (
    (territory_id, territory, payer_display_id, payer_display_name),
    (payer_display_id, payer_display_name),
    (territory_id, territory),
    ()
  )
)

, final_with_lives AS (
  SELECT r.*,
         COALESCE(pu.TOTAL_HCPS,0) AS TOTAL_HCPS,
         COALESCE(pu.TOTAL_HCOS,0) AS TOTAL_HCOS,
         COALESCE(t.total_lives,0) AS total_lives
  FROM rollup_metrics r
  LEFT JOIN provider_universe_rollup pu
    ON r.rollup_level = pu.rollup_level
   AND r.territory_id = pu.territory_id
   AND r.payer_display_id = pu.payer_display_id
  LEFT JOIN total_lives t
    ON r.rollup_level = t.rollup_level
   AND r.territory_id = t.territory_id
   AND r.payer_display_id = t.payer_display_id
)

, final_with_share AS (
  SELECT f.*,
    100.0 * f.total_lives /
    NULLIF(
      CASE
        WHEN f.rollup_level = 'TERRITORY_PAYER' THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level, f.territory_id)
        WHEN f.rollup_level = 'PAYER_ALL_TERRITORY' THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
        WHEN f.rollup_level = 'TERRITORY_ALL_PAYER' THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level, f.territory_id) -- territory all payer: share of territory across payers
        WHEN f.rollup_level = 'NATIONAL' THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
      END, 0) AS payer_market_share_pct
  FROM final_with_lives f
)

, final_with_rank AS (
  SELECT s.*,
     ROW_NUMBER() OVER (
       PARTITION BY
         CASE WHEN s.rollup_level = 'TERRITORY_PAYER' THEN CONCAT(s.rollup_level, '||', s.territory_id)
              ELSE s.rollup_level END
       ORDER BY s.payer_market_share_pct DESC, s.total_lives DESC, s.payer_display_id
     ) AS payer_rank
  FROM final_with_share s
)

SELECT DISTINCT
  territory_id,
  territory_name,
  payer_display_id  AS payer_id,
  payer_display_name AS payer_name,
  payer_group,
  parent_id,
  parent_name,
  ROUND(COALESCE(payer_market_share_pct,0),6) AS payer_market_share_pct,
  payer_rank,
  total_lives,
  total_elaprase_patients,
  medicare_patients,
  medicaid_patients,
  commercial_patients,
  other_patients,
  NEW_ELAPRASE_PATIENTS_R1M,
  NEW_ELAPRASE_PATIENTS_R3M,
  AGE_LT_5_YRS,
  AGE_5_TO_10_YRS,
  AGE_11_TO_18_YRS,
  AGE_GT_18_YRS,
  TOTAL_PRIMARY_HCPS,
  TOTAL_HCPS,
  TOTAL_PRIMARY_HCOS,
  TOTAL_HCOS,
  TOTAL_CLAIMS,
  PHARMACY_TOTAL_CLAIMS,
  APPROVED_FILLS,
  REJECTED_FILLS,
  REVERSED_FILLS,
  ELAPRASE_APPROVAL_RATE,
  ELAPRASE_REJECTION_RATE,
  ELAPRASE_REVERSED_RATE,
  PIE_COMPLETED,
  ACCOUNT_DIRECTOR,
  rollup_level
FROM final_with_rank
ORDER BY rollup_level, territory_name, payer_name;

-- ============================================================
-- 8) HCP / HCO detail tables (use payer_display_* from all_patient_claims_expanded)
--    These will now reflect rolled-up payer names automatically.
-- ============================================================
CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL AS
SELECT
  COALESCE(territory_id, 'ALL Territories') AS territory_id,
  COALESCE(territory_name,'All Territories') AS territory_name,
  COALESCE(payer_display_id,'ALL Payers') AS payer_id,
  COALESCE(payer_display_name,'All Payers') AS payer_name,
  COALESCE(hcp_npi,'ALL HCPs') AS hcp_npi,
  COALESCE(hcp_name,'All HCPs') AS hcp_name,
  COALESCE(hcp_specialty,'All HCPs') AS hcp_specialty,
  COALESCE(hco_npi,'ALL HCOs') AS hco_npi,
  CASE WHEN GROUPING(hco_npi)=1 THEN 'All HCOs' ELSE MAX(hco_name) END AS hco_name,
  COUNT(DISTINCT patient_id) AS patient_count,
  COUNT(DISTINCT claim_id) AS claims_count,
  COUNT(DISTINCT hco_npi) AS total_hcos,
  MAX(fill_date) AS last_treatment_date,
  CASE
    WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=0 THEN 'TERRITORY_PAYER'
    WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=0 THEN 'PAYER_ALL_TERRITORY'
    WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=1 THEN 'TERRITORY_ALL_PAYER'
    WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=1 THEN 'NATIONAL'
  END AS rollup_level
FROM all_patient_claims_expanded
GROUP BY GROUPING SETS (
  (territory_id, territory_name, payer_display_id, payer_display_name, hcp_npi, hcp_name, hcp_specialty, hco_npi),
  (payer_display_id, payer_display_name, hcp_npi, hcp_name, hcp_specialty, hco_npi),
  (territory_id, territory_name),
  ()
)
ORDER BY rollup_level, territory_name, payer_name, hcp_npi;


CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL AS
SELECT
  COALESCE(territory_id,'ALL Territories') AS territory_id,
  COALESCE(territory_name,'All Territories') AS territory_name,
  COALESCE(payer_display_id,'ALL Payers') AS payer_id,
  COALESCE(payer_display_name,'All Payers') AS payer_name,
  CASE WHEN GROUPING(hco_npi)=1 THEN 'ALL HCOs' ELSE hco_npi END AS hco_npi,
  CASE WHEN GROUPING(hco_npi)=1 THEN 'All HCOs' ELSE MAX(hco_name) END AS hco_name,
  COUNT(DISTINCT patient_id) AS patient_count,
  COUNT(DISTINCT claim_id) AS claims_count,
  MAX(fill_date) AS last_treatment_date,
  CASE
    WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=0 THEN 'TERRITORY_PAYER'
    WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=0 THEN 'PAYER_ALL_TERRITORY'
    WHEN GROUPING(territory_id)=0 AND GROUPING(payer_display_id)=1 THEN 'TERRITORY_ALL_PAYER'
    WHEN GROUPING(territory_id)=1 AND GROUPING(payer_display_id)=1 THEN 'NATIONAL'
  END AS rollup_level
FROM all_patient_claims_expanded
GROUP BY GROUPING SETS (
  (territory_id, territory_name, payer_display_id, payer_display_name, hco_npi),
  (payer_display_id, payer_display_name, hco_npi),
  (territory_id, territory_name),
  ()
)
ORDER BY rollup_level, territory_name, payer_name, hco_npi;

-- ============================================================
-- 9) Important QC checks (run and review)
--    - national patient parity
--    - market share sum per territory ~100
--    - patient counts <= lives
--    - payer names identical across payer360_master and HCP/HCO detail
-- ============================================================
-- QC-A: National parity: payer360 national vs count distinct in patient master
SELECT
  (SELECT COUNT(DISTINCT patient_id) FROM payer_master_patient_level) AS patient_master_distinct,
  (SELECT total_elaprase_patients FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master WHERE rollup_level='NATIONAL') AS payer360_national,
  ((SELECT COUNT(DISTINCT patient_id) FROM payer_master_patient_level) - (SELECT total_elaprase_patients FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master WHERE rollup_level='NATIONAL')) AS diff;

-- QC-B: Market share sums (territory level should sum ~100)
SELECT territory_id, ROUND(SUM(payer_market_share_pct),2) AS total_share
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
WHERE rollup_level = 'TERRITORY_PAYER'
GROUP BY territory_id
HAVING ROUND(SUM(payer_market_share_pct),2) NOT BETWEEN 99.9 AND 100.1;

-- QC-C: Patient counts should not exceed total_lives
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master WHERE total_elaprase_patients > total_lives;

-- QC-D: Unique payer names parity
SELECT distinct payer_name FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
MINUS
SELECT distinct payer_name FROM cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL
LIMIT 50;

-- QC-E: HCP/HCO counts at national level vs payer360
SELECT
  m.total_elaprase_patients AS payer360_patients,
  h.patient_count AS hcp_patients,
  c.patient_count AS hco_patients
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master m
LEFT JOIN cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL h
  ON m.rollup_level = h.rollup_level AND m.territory_id = h.territory_id AND m.payer_id = h.payer_id
LEFT JOIN cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL c
  ON m.rollup_level = c.rollup_level AND m.territory_id = c.territory_id AND m.payer_id = c.payer_id
WHERE m.rollup_level = 'NATIONAL';

-- QC-F: Quick distinct payer list check (should be identical)
SELECT 'payer360_only', payer_name FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
EXCEPT
SELECT 'hcp_only', payer_name FROM cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL
LIMIT 20;

-- ============================================================
-- End of script
-- ============================================================

In [0]:
%sql
    SELECT COUNT(DISTINCT patient_id)
    FROM (

        -- Medical (Elaprase NDC)
        SELECT DISTINCT
            PATIENT_ID                                  AS PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
            BILLING_NPI                                 AS HCO_NPI,
            NDC11                                       AS CODE,
            MEDICAL_EVENT_ID                            AS EVENT_ID,
            SERVICE_DATE                                AS FILL_DATE,
            PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
            KH_PLAN_ID                                  AS KH_PLAN,
            NULL                                        AS PHARMACY_CHANNEL,
            'MEDICAL_EVENTS'                            AS TABLE_NAME
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
        and SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

        UNION

        -- Pharmacy (Elaprase NDC | Paid only)
        SELECT DISTINCT
            PATIENT_ID                                  AS PATIENT_ID,
            PRESCRIBER_NPI                              AS HCP_NPI,
            PHARMACY_NPI                                AS HCO_NPI,
            NDC11                                       AS CODE,
            PHARMACY_EVENT_ID                           AS EVENT_ID,
            FILL_DATE                                   AS FILL_DATE,
            NULL                                        AS PLACE_OF_SERVICE,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
            PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
            'PHARMACY_EVENTS'                           AS TABLE_NAME
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30'

        UNION

        -- Medical (Elaprase Procedure codes)
        SELECT DISTINCT
            PATIENT_ID                                  AS PATIENT_ID,
            RENDERING_NPI                               AS HCP_NPI,
            BILLING_NPI                                 AS HCO_NPI,
            PROCEDURE_CODE                              AS CODE,
            MEDICAL_EVENT_ID                            AS EVENT_ID,
            SERVICE_DATE                                AS FILL_DATE,
            PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
            KH_PLAN_ID                                  AS KH_PLAN,
            NULL                                        AS PHARMACY_CHANNEL,
            'MEDICAL_EVENTS'                            AS TABLE_NAME
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN (
            '99601','99602','96365','96366','J1743','S9357','S9379',
            '38206','38230','38232','38240','38241','38242','38243','38250'
        ) and SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

    )

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
WITH base AS (
    SELECT *
    FROM (

        -- Medical (Elaprase NDC)
        SELECT DISTINCT
            PATIENT_ID                                  AS PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
            BILLING_NPI                                 AS HCO_NPI,
            NDC11                                       AS CODE,
            MEDICAL_EVENT_ID                            AS EVENT_ID,
            SERVICE_DATE                                AS FILL_DATE,
            PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
            KH_PLAN_ID                                  AS KH_PLAN,
            NULL                                        AS PHARMACY_CHANNEL,
            'MEDICAL_EVENTS'                            AS TABLE_NAME
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
        and SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

        UNION

        -- Pharmacy (Elaprase NDC | Paid only)
        SELECT DISTINCT
            PATIENT_ID                                  AS PATIENT_ID,
            PRESCRIBER_NPI                              AS HCP_NPI,
            PHARMACY_NPI                                AS HCO_NPI,
            NDC11                                       AS CODE,
            PHARMACY_EVENT_ID                           AS EVENT_ID,
            FILL_DATE                                   AS FILL_DATE,
            NULL                                        AS PLACE_OF_SERVICE,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
            PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
            'PHARMACY_EVENTS'                           AS TABLE_NAME
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30'

        UNION

        -- Medical (Elaprase Procedure codes)
        SELECT DISTINCT
            PATIENT_ID                                  AS PATIENT_ID,
            RENDERING_NPI                               AS HCP_NPI,
            BILLING_NPI                                 AS HCO_NPI,
            PROCEDURE_CODE                              AS CODE,
            MEDICAL_EVENT_ID                            AS EVENT_ID,
            SERVICE_DATE                                AS FILL_DATE,
            PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
            KH_PLAN_ID                                  AS KH_PLAN,
            NULL                                        AS PHARMACY_CHANNEL,
            'MEDICAL_EVENTS'                            AS TABLE_NAME
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN (
            '99601','99602','96365','96366','J1743','S9357','S9379',
            '38206','38230','38232','38240','38241','38242','38243','38250'
        ) and SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

    ) t
),

prov AS (
    SELECT
        TRIM(CAST(NPI AS STRING)) AS npi_str,
        MAX(
            TRY_CAST(
                SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP, ''), '[^0-9]', ''), 1, 5) AS BIGINT
            )
        ) AS zip5_int
    FROM com_edp_prd.com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
    GROUP BY TRIM(CAST(NPI AS STRING))
)

SELECT
    b.*,
    COALESCE(TRY_CAST(z.territory_id AS BIGINT), -1) AS territory_id,
    COALESCE(z.territory_name, 'UNKNOWN')            AS territory_name,
    COALESCE(TRY_CAST(z.region_id AS BIGINT), -1)    AS region_id,
    COALESCE(z.region_name, 'UNKNOWN')               AS region_name
FROM base b
LEFT JOIN prov p
    ON TRIM(CAST(b.HCP_NPI AS STRING)) = p.npi_str
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON p.zip5_int = z.zipcode;


In [0]:
%sql
SELECT
  territory_id,
  territory_name,
  COUNT(DISTINCT PATIENT_ID) AS TOTAL_PATIENTS
FROM MPSII_TREATMENT_TABLE
GROUP BY territory_id, territory_name
ORDER BY TOTAL_PATIENTS DESC;


In [0]:
%sql
SELECT
 * from com_edp_prd.com_raw.kom_patient_demographics where patient_id = 'ZDZPPDK2';


--  select patient_id, count(distinct patient_yob) from com_edp_prd.com_raw.kom_patient_demographics group by 1 order by 2 desc;


In [0]:
%sql
SELECT distinct INSURANCE_GROUP
FROM com_edp_prd.com_raw.kom_plans ;

In [0]:
%sql
SELECT * FROM com_edp_prd.reltio_in_out.master_target_hcp_and_hco   ;  

In [0]:
%sql
SELECT * FROM cmpa_insights_internal_schema.reference_file_0109; 

In [0]:
%sql
select 
-- count(distinct PATIENT_ID), count(*), max(latest_claim_date) 
distinct patient_gender
-- *
from com_edp_prd.cmpa_insights_internal_schema.patient360

In [0]:
%sql
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file

In [0]:
%sql
SELECT COUNT(DISTINCT PATIENT_ID) FROM com_edp_prd.cmpa_insights_internal_schema.patient360;

In [0]:
%sql
SELECT DISTINCT PATIENT_ID FROM com_edp_prd.cmpa_insights_internal_schema.patient360;

In [0]:
%sql
select * from com_edp_prd.dcr.dcr_vod_status where vcrm_dcr_id in ('V7E000000001001',
'V7E000000001002',
'V7E000000001003',
'V7E000000001004',
'V7E000000002001',
'V7E000000003001',
'V7E000000003002',
'V7E000000003003',
'V7E000000003004',
'V7E000000003005',
'V7E000000003006',
'V7E000000003007',
'V7E000000003008',
'V7E000000003009',
'V7E000000003010',
'V7E000000004001',
'V7E000000004002',
'V7E000000004003',
'V7E000000005001',
'V7E000000005002',
'V7E000000005003',
'V7E000000005004',
'V7E000000006001',
'V7E000000006002',
'V7E000000006003',
'V7E000000006004',
'V7E000000006005',
'V7E000000006006',
'V7E000000006007',
'V7E000000006008',
'V7E000000007001',
'V7E000000007002',
'V7E000000007003',
'V7E000000007004',
'V7E000000007005',
'V7E000000007006',
'V7E000000007007',
'V7E000000008001',
'V7E000000008002',
'V7E000000008003',
'V7E000000009001',
'V7E00000000A001',
'V7E00000000A002',
'V7E00000000A003',
'V7E00000000A004',
'V7E000000009002',
'V7E000000009003',
'V7E000000009004',
'V7E000000009005',
'V7E000000009006',
'V7E00000000A008',
'V7E00000000A009',
'V7E00000000A010',
'V7E00000000A011',
'V7E00000000A012',
'V7E000000009019',
'V7E000000009020',
'V7E000000009021',
'V7E00000000A013',
'V7E00000000A014',
'V7E00000000A015',
'V7E00000000A016',
'V7E00000000A017',
'V7E00000000A018',
'V7E00000000A019',
'V7E00000000A020',
'V7E00000000A021',
'V7E00000000A022',
'V7E00000000A023',
'V7E00000000A024',
'V7E00000000B001',
'V7E00000000B002',
'V7E00000000B003',
'V7E00000000B004',
'V7E00000000B005',
'V7E00000000B006',
'V7E00000000B007',
'V7E00000000B008',
'V7E00000000B009',
'V7E00000000B010',
'V7E00000000C001',
'V7E00000000C002',
'V7E00000000C003',
'V7E00000000C004',
'V7E00000000C005',
'V7E00000000D001',
'V7E00000000D002',
'V7E00000000D003',
'V7E00000000D004',
'V7E00000000D005',
'V7E00000000D006',
'V7E00000000D007',
'V7E00000000D008',
'V7E00000000D009',
'V7E00000000E001',
'V7E00000000D010',
'V7E00000000D011',
'V7E00000000D012',
'V7E00000000D013',
'V7E00000000D014',
'V7E00000000D015',
'V7E00000000D016',
'V7E00000000D017',
'V7E00000000D018',
'V7E00000000F001',
'V7E00000000G001',
'V7E00000000G002',
'V7E00000000G003',
'V7E00000000G004',
'V7E00000000G005',
'V7E00000000G006',
'V7E00000000G007',
'V7E00000000H001',
'V7E00000000H002',
'V7E00000000H003',
'V7E00000000I001',
'V7E00000000I002',
'V7E00000000I003',
'V7E00000000I004',
'V7E00000000I005',
'V7E00000000I006',
'V7E00000000H004',
'V7E00000000H005',
'V7E00000000H006',
'V7E00000000H007',
'V7E00000000H008',
'V7E00000000I007',
'V7E00000000I008',
'V7E00000000I009',
'V7E00000000I010',
'V7E00000000I011',
'V7E00000000I012',
'V7E00000000I013',
'V7E00000000H009',
'V7E00000000H010',
'V7E00000000H011',
'V7E00000000H012',
'V7E00000000H013',
'V7E00000000H014',
'V7E00000000H015',
'V7E00000000H016',
'V7E00000000H017',
'V7E00000000H018',
'V7E00000000H019',
'V7E00000000H020',
'V7E00000000H021',
'V7E00000000H022',
'V7E00000000H023',
'V7E00000000I014',
'V7E00000000I015',
'V7E00000000I016',
'V7E00000000I017',
'V7E00000000I018',
'V7E00000000I019',
'V7E00000000H024',
'V7E00000000H025',
'V7E00000000H026',
'V7E00000000H027',
'V7E00000000I020',
'V7E00000000H028',
'V7E00000000H029',
'V7E00000000H030',
'V7E00000000H031',
'V7E00000000H032',
'V7E00000000H033',
'V7E00000000H034',
'V7E00000000H035',
'V7E00000000H036',
'V7E00000000H037',
'V7E00000000H038',
'V7E00000000H039',
'V7E00000000H040',
'V7E00000000H041',
'V7E00000000H042',
'V7E00000000H043',
'V7E00000000H044',
'V7E00000000J001',
'V7E00000000J002',
'V7E00000000J003',
'V7E00000000J004',
'V7E00000000J005',
'V7E00000000J006',
'V7E00000000J007',
'V7E00000000K001',
'V7E00000000K002',
'V7E00000000K003',
'V7E00000000K004',
'V7E00000000K005',
'V7E00000000K006',
'V7E00000000K007',
'V7E00000000K008',
'V7E00000000L001',
'V7E00000000L002',
'V7E00000000L003',
'V7E00000000L004',
'V7E00000000L005',
'V7E00000000L006',
'V7E00000000L007',
'V7E00000000L008',
'V7E00000000L009',
'V7E00000000L010',
'V7E00000000L011',
'V7E00000000M001',
'V7E00000000M002',
'V7E00000000M003',
'V7E00000000M004',
'V7E00000000M005',
'V7E00000000N001',
'V7E00000000N002',
'V7E00000000N003',
'V7E00000000N004',
'V7E00000000N005',
'V7E00000000N006',
'V7E00000000N007',
'V7E00000000N008',
'V7E00000000N009',
'V7E00000000N010',
'V7E00000000N011',
'V7E00000000N012',
'V7E00000000N013',
'V7E00000000N014',
'V7E00000000O001',
'V7E00000000O002',
'V7E00000000O003',
'V7E00000000O004',
'V7E00000000O005',
'V7E00000000N015',
'V7E00000000N016',
'V7E00000000N017',
'V7E00000000N018',
'V7E00000000N019',
'V7E00000000N020',
'V7E00000000P001',
'V7E00000000P002',
'V7E00000000P003',
'V7E00000000Q001',
'V7E00000000Q002',
'V7E00000000Q003',
'V7E00000000R001',
'V7E00000000R002',
'V7E00000000R003',
'V7E00000000R004',
'V7E00000000R005',
'V7E00000000S001',
'V7E00000000S002',
'V7E00000000S003',
'V7E00000000S004',
'V7E00000000S005',
'V7E00000000S006',
'V7E00000000S007',
'V7E00000000S008',
'V7E00000000S009',
'V7E00000000S010',
'V7E00000000S011',
'V7E00000000S012',
'V7E00000000T001',
'V7E00000000T002',
'V7E00000000T003',
'V7E00000000T004',
'V7E00000000T005',
'V7E00000000T006',
'V7E00000000T007',
'V7E00000000T008',
'V7E00000000T009',
'V7E00000000T010',
'V7E00000000T011',
'V7E00000000T012',
'V7E00000000T013',
'V7E00000000T014',
'V7E00000000T015',
'V7E00000000T016',
'V7E00000000T017',
'V7E00000000T018',
'V7E00000000T019',
'V7E00000000T020',
'V7E00000000T021',
'V7E00000000T022',
'V7E00000000T023',
'V7E00000000T024',
'V7E00000000T025',
'V7E00000000T026',
'V7E00000000T027',
'V7E00000000T028',
'V7E00000000T029',
'V7E00000000T030',
'V7E00000000T031',
'V7E00000000T032',
'V7E00000000T033',
'V7E00000000T034',
'V7E00000000T035',
'V7E00000000T036',
'V7E00000000T037',
'V7E00000000T038',
'V7E00000000T039',
'V7E00000000T040',
'V7E00000000T041',
'V7E00000000T042',
'V7E00000000T043',
'V7E00000000T044',
'V7E00000000T045',
'V7E00000000T046',
'V7E00000000T047',
'V7E00000000S013',
'V7E00000000U001',
'V7E00000000U002',
'V7E00000000U003',
'V7E00000000V001',
'V7E00000000V002',
'V7E00000000V003',
'V7E00000000V004',
'V7E00000000V005',
'V7E00000000V006',
'V7E00000000V007',
'V7E00000000V008',
'V7E00000000W001',
'V7E00000000W002',
'V7E00000000W003',
'V7E00000000W004',
'V7E00000000W005',
'V7E00000000W006',
'V7E00000000V009',
'V7E00000000V010',
'V7E00000000V011',
'V7E00000000W007',
'V7E00000000V012',
'V7E00000000W008',
'V7E00000000W009',
'V7E00000000W010',
'V7E00000000W011',
'V7E00000000V013',
'V7E00000000V014',
'V7E00000000V015',
'V7E00000000V016'
)


In [0]:

%sql
----- found changed values- com_edp_prd.dcr.vw_dcr_account_edit
select * from com_edp_prd.dcr.vw_dcr_address_edit 
where dcr_id in ('V7E000000001001',
'V7E000000001002',
'V7E000000001003',
'V7E000000001004',
'V7E000000002001',
'V7E000000003001',
'V7E000000003002',
'V7E000000003003',
'V7E000000003004',
'V7E000000003005',
'V7E000000003006',
'V7E000000003007',
'V7E000000003008',
'V7E000000003009',
'V7E000000003010',
'V7E000000004001',
'V7E000000004002',
'V7E000000004003',
'V7E000000005001',
'V7E000000005002',
'V7E000000005003',
'V7E000000005004',
'V7E000000006001',
'V7E000000006002',
'V7E000000006003',
'V7E000000006004',
'V7E000000006005',
'V7E000000006006',
'V7E000000006007',
'V7E000000006008',
'V7E000000007001',
'V7E000000007002',
'V7E000000007003',
'V7E000000007004',
'V7E000000007005',
'V7E000000007006',
'V7E000000007007',
'V7E000000008001',
'V7E000000008002',
'V7E000000008003',
'V7E000000009001',
'V7E00000000A001',
'V7E00000000A002',
'V7E00000000A003',
'V7E00000000A004',
'V7E000000009002',
'V7E000000009003',
'V7E000000009004',
'V7E000000009005',
'V7E000000009006',
'V7E00000000A008',
'V7E00000000A009',
'V7E00000000A010',
'V7E00000000A011',
'V7E00000000A012',
'V7E000000009019',
'V7E000000009020',
'V7E000000009021',
'V7E00000000A013',
'V7E00000000A014',
'V7E00000000A015',
'V7E00000000A016',
'V7E00000000A017',
'V7E00000000A018',
'V7E00000000A019',
'V7E00000000A020',
'V7E00000000A021',
'V7E00000000A022',
'V7E00000000A023',
'V7E00000000A024',
'V7E00000000B001',
'V7E00000000B002',
'V7E00000000B003',
'V7E00000000B004',
'V7E00000000B005',
'V7E00000000B006',
'V7E00000000B007',
'V7E00000000B008',
'V7E00000000B009',
'V7E00000000B010',
'V7E00000000C001',
'V7E00000000C002',
'V7E00000000C003',
'V7E00000000C004',
'V7E00000000C005',
'V7E00000000D001',
'V7E00000000D002',
'V7E00000000D003',
'V7E00000000D004',
'V7E00000000D005',
'V7E00000000D006',
'V7E00000000D007',
'V7E00000000D008',
'V7E00000000D009',
'V7E00000000E001',
'V7E00000000D010',
'V7E00000000D011',
'V7E00000000D012',
'V7E00000000D013',
'V7E00000000D014',
'V7E00000000D015',
'V7E00000000D016',
'V7E00000000D017',
'V7E00000000D018',
'V7E00000000F001',
'V7E00000000G001',
'V7E00000000G002',
'V7E00000000G003',
'V7E00000000G004',
'V7E00000000G005',
'V7E00000000G006',
'V7E00000000G007',
'V7E00000000H001',
'V7E00000000H002',
'V7E00000000H003',
'V7E00000000I001',
'V7E00000000I002',
'V7E00000000I003',
'V7E00000000I004',
'V7E00000000I005',
'V7E00000000I006',
'V7E00000000H004',
'V7E00000000H005',
'V7E00000000H006',
'V7E00000000H007',
'V7E00000000H008',
'V7E00000000I007',
'V7E00000000I008',
'V7E00000000I009',
'V7E00000000I010',
'V7E00000000I011',
'V7E00000000I012',
'V7E00000000I013',
'V7E00000000H009',
'V7E00000000H010',
'V7E00000000H011',
'V7E00000000H012',
'V7E00000000H013',
'V7E00000000H014',
'V7E00000000H015',
'V7E00000000H016',
'V7E00000000H017',
'V7E00000000H018',
'V7E00000000H019',
'V7E00000000H020',
'V7E00000000H021',
'V7E00000000H022',
'V7E00000000H023',
'V7E00000000I014',
'V7E00000000I015',
'V7E00000000I016',
'V7E00000000I017',
'V7E00000000I018',
'V7E00000000I019',
'V7E00000000H024',
'V7E00000000H025',
'V7E00000000H026',
'V7E00000000H027',
'V7E00000000I020',
'V7E00000000H028',
'V7E00000000H029',
'V7E00000000H030',
'V7E00000000H031',
'V7E00000000H032',
'V7E00000000H033',
'V7E00000000H034',
'V7E00000000H035',
'V7E00000000H036',
'V7E00000000H037',
'V7E00000000H038',
'V7E00000000H039',
'V7E00000000H040',
'V7E00000000H041',
'V7E00000000H042',
'V7E00000000H043',
'V7E00000000H044',
'V7E00000000J001',
'V7E00000000J002',
'V7E00000000J003',
'V7E00000000J004',
'V7E00000000J005',
'V7E00000000J006',
'V7E00000000J007',
'V7E00000000K001',
'V7E00000000K002',
'V7E00000000K003',
'V7E00000000K004',
'V7E00000000K005',
'V7E00000000K006',
'V7E00000000K007',
'V7E00000000K008',
'V7E00000000L001',
'V7E00000000L002',
'V7E00000000L003',
'V7E00000000L004',
'V7E00000000L005',
'V7E00000000L006',
'V7E00000000L007',
'V7E00000000L008',
'V7E00000000L009',
'V7E00000000L010',
'V7E00000000L011',
'V7E00000000M001',
'V7E00000000M002',
'V7E00000000M003',
'V7E00000000M004',
'V7E00000000M005',
'V7E00000000N001',
'V7E00000000N002',
'V7E00000000N003',
'V7E00000000N004',
'V7E00000000N005',
'V7E00000000N006',
'V7E00000000N007',
'V7E00000000N008',
'V7E00000000N009',
'V7E00000000N010',
'V7E00000000N011',
'V7E00000000N012',
'V7E00000000N013',
'V7E00000000N014',
'V7E00000000O001',
'V7E00000000O002',
'V7E00000000O003',
'V7E00000000O004',
'V7E00000000O005',
'V7E00000000N015',
'V7E00000000N016',
'V7E00000000N017',
'V7E00000000N018',
'V7E00000000N019',
'V7E00000000N020',
'V7E00000000P001',
'V7E00000000P002',
'V7E00000000P003',
'V7E00000000Q001',
'V7E00000000Q002',
'V7E00000000Q003',
'V7E00000000R001',
'V7E00000000R002',
'V7E00000000R003',
'V7E00000000R004',
'V7E00000000R005',
'V7E00000000S001',
'V7E00000000S002',
'V7E00000000S003',
'V7E00000000S004',
'V7E00000000S005',
'V7E00000000S006',
'V7E00000000S007',
'V7E00000000S008',
'V7E00000000S009',
'V7E00000000S010',
'V7E00000000S011',
'V7E00000000S012',
'V7E00000000T001',
'V7E00000000T002',
'V7E00000000T003',
'V7E00000000T004',
'V7E00000000T005',
'V7E00000000T006',
'V7E00000000T007',
'V7E00000000T008',
'V7E00000000T009',
'V7E00000000T010',
'V7E00000000T011',
'V7E00000000T012',
'V7E00000000T013',
'V7E00000000T014',
'V7E00000000T015',
'V7E00000000T016',
'V7E00000000T017',
'V7E00000000T018',
'V7E00000000T019',
'V7E00000000T020',
'V7E00000000T021',
'V7E00000000T022',
'V7E00000000T023',
'V7E00000000T024',
'V7E00000000T025',
'V7E00000000T026',
'V7E00000000T027',
'V7E00000000T028',
'V7E00000000T029',
'V7E00000000T030',
'V7E00000000T031',
'V7E00000000T032',
'V7E00000000T033',
'V7E00000000T034',
'V7E00000000T035',
'V7E00000000T036',
'V7E00000000T037',
'V7E00000000T038',
'V7E00000000T039',
'V7E00000000T040',
'V7E00000000T041',
'V7E00000000T042',
'V7E00000000T043',
'V7E00000000T044',
'V7E00000000T045',
'V7E00000000T046',
'V7E00000000T047',
'V7E00000000S013',
'V7E00000000U001',
'V7E00000000U002',
'V7E00000000U003',
'V7E00000000V001',
'V7E00000000V002',
'V7E00000000V003',
'V7E00000000V004',
'V7E00000000V005',
'V7E00000000V006',
'V7E00000000V007',
'V7E00000000V008',
'V7E00000000W001',
'V7E00000000W002',
'V7E00000000W003',
'V7E00000000W004',
'V7E00000000W005',
'V7E00000000W006',
'V7E00000000V009',
'V7E00000000V010',
'V7E00000000V011',
'V7E00000000W007',
'V7E00000000V012',
'V7E00000000W008',
'V7E00000000W009',
'V7E00000000W010',
'V7E00000000W011',
'V7E00000000V013',
'V7E00000000V014',
'V7E00000000V015',
'V7E00000000V016'
)

In [0]:

%sql
----- found changed values- com_edp_prd.dcr.vw_dcr_account_edit
select * from com_edp_prd.dcr.vw_dcr_account_edit
where dcr_id in ('V7E000000001001',
'V7E000000001002',
'V7E000000001003',
'V7E000000001004',
'V7E000000002001',
'V7E000000003001',
'V7E000000003002',
'V7E000000003003',
'V7E000000003004',
'V7E000000003005',
'V7E000000003006',
'V7E000000003007',
'V7E000000003008',
'V7E000000003009',
'V7E000000003010',
'V7E000000004001',
'V7E000000004002',
'V7E000000004003',
'V7E000000005001',
'V7E000000005002',
'V7E000000005003',
'V7E000000005004',
'V7E000000006001',
'V7E000000006002',
'V7E000000006003',
'V7E000000006004',
'V7E000000006005',
'V7E000000006006',
'V7E000000006007',
'V7E000000006008',
'V7E000000007001',
'V7E000000007002',
'V7E000000007003',
'V7E000000007004',
'V7E000000007005',
'V7E000000007006',
'V7E000000007007',
'V7E000000008001',
'V7E000000008002',
'V7E000000008003',
'V7E000000009001',
'V7E00000000A001',
'V7E00000000A002',
'V7E00000000A003',
'V7E00000000A004',
'V7E000000009002',
'V7E000000009003',
'V7E000000009004',
'V7E000000009005',
'V7E000000009006',
'V7E00000000A008',
'V7E00000000A009',
'V7E00000000A010',
'V7E00000000A011',
'V7E00000000A012',
'V7E000000009019',
'V7E000000009020',
'V7E000000009021',
'V7E00000000A013',
'V7E00000000A014',
'V7E00000000A015',
'V7E00000000A016',
'V7E00000000A017',
'V7E00000000A018',
'V7E00000000A019',
'V7E00000000A020',
'V7E00000000A021',
'V7E00000000A022',
'V7E00000000A023',
'V7E00000000A024',
'V7E00000000B001',
'V7E00000000B002',
'V7E00000000B003',
'V7E00000000B004',
'V7E00000000B005',
'V7E00000000B006',
'V7E00000000B007',
'V7E00000000B008',
'V7E00000000B009',
'V7E00000000B010',
'V7E00000000C001',
'V7E00000000C002',
'V7E00000000C003',
'V7E00000000C004',
'V7E00000000C005',
'V7E00000000D001',
'V7E00000000D002',
'V7E00000000D003',
'V7E00000000D004',
'V7E00000000D005',
'V7E00000000D006',
'V7E00000000D007',
'V7E00000000D008',
'V7E00000000D009',
'V7E00000000E001',
'V7E00000000D010',
'V7E00000000D011',
'V7E00000000D012',
'V7E00000000D013',
'V7E00000000D014',
'V7E00000000D015',
'V7E00000000D016',
'V7E00000000D017',
'V7E00000000D018',
'V7E00000000F001',
'V7E00000000G001',
'V7E00000000G002',
'V7E00000000G003',
'V7E00000000G004',
'V7E00000000G005',
'V7E00000000G006',
'V7E00000000G007',
'V7E00000000H001',
'V7E00000000H002',
'V7E00000000H003',
'V7E00000000I001',
'V7E00000000I002',
'V7E00000000I003',
'V7E00000000I004',
'V7E00000000I005',
'V7E00000000I006',
'V7E00000000H004',
'V7E00000000H005',
'V7E00000000H006',
'V7E00000000H007',
'V7E00000000H008',
'V7E00000000I007',
'V7E00000000I008',
'V7E00000000I009',
'V7E00000000I010',
'V7E00000000I011',
'V7E00000000I012',
'V7E00000000I013',
'V7E00000000H009',
'V7E00000000H010',
'V7E00000000H011',
'V7E00000000H012',
'V7E00000000H013',
'V7E00000000H014',
'V7E00000000H015',
'V7E00000000H016',
'V7E00000000H017',
'V7E00000000H018',
'V7E00000000H019',
'V7E00000000H020',
'V7E00000000H021',
'V7E00000000H022',
'V7E00000000H023',
'V7E00000000I014',
'V7E00000000I015',
'V7E00000000I016',
'V7E00000000I017',
'V7E00000000I018',
'V7E00000000I019',
'V7E00000000H024',
'V7E00000000H025',
'V7E00000000H026',
'V7E00000000H027',
'V7E00000000I020',
'V7E00000000H028',
'V7E00000000H029',
'V7E00000000H030',
'V7E00000000H031',
'V7E00000000H032',
'V7E00000000H033',
'V7E00000000H034',
'V7E00000000H035',
'V7E00000000H036',
'V7E00000000H037',
'V7E00000000H038',
'V7E00000000H039',
'V7E00000000H040',
'V7E00000000H041',
'V7E00000000H042',
'V7E00000000H043',
'V7E00000000H044',
'V7E00000000J001',
'V7E00000000J002',
'V7E00000000J003',
'V7E00000000J004',
'V7E00000000J005',
'V7E00000000J006',
'V7E00000000J007',
'V7E00000000K001',
'V7E00000000K002',
'V7E00000000K003',
'V7E00000000K004',
'V7E00000000K005',
'V7E00000000K006',
'V7E00000000K007',
'V7E00000000K008',
'V7E00000000L001',
'V7E00000000L002',
'V7E00000000L003',
'V7E00000000L004',
'V7E00000000L005',
'V7E00000000L006',
'V7E00000000L007',
'V7E00000000L008',
'V7E00000000L009',
'V7E00000000L010',
'V7E00000000L011',
'V7E00000000M001',
'V7E00000000M002',
'V7E00000000M003',
'V7E00000000M004',
'V7E00000000M005',
'V7E00000000N001',
'V7E00000000N002',
'V7E00000000N003',
'V7E00000000N004',
'V7E00000000N005',
'V7E00000000N006',
'V7E00000000N007',
'V7E00000000N008',
'V7E00000000N009',
'V7E00000000N010',
'V7E00000000N011',
'V7E00000000N012',
'V7E00000000N013',
'V7E00000000N014',
'V7E00000000O001',
'V7E00000000O002',
'V7E00000000O003',
'V7E00000000O004',
'V7E00000000O005',
'V7E00000000N015',
'V7E00000000N016',
'V7E00000000N017',
'V7E00000000N018',
'V7E00000000N019',
'V7E00000000N020',
'V7E00000000P001',
'V7E00000000P002',
'V7E00000000P003',
'V7E00000000Q001',
'V7E00000000Q002',
'V7E00000000Q003',
'V7E00000000R001',
'V7E00000000R002',
'V7E00000000R003',
'V7E00000000R004',
'V7E00000000R005',
'V7E00000000S001',
'V7E00000000S002',
'V7E00000000S003',
'V7E00000000S004',
'V7E00000000S005',
'V7E00000000S006',
'V7E00000000S007',
'V7E00000000S008',
'V7E00000000S009',
'V7E00000000S010',
'V7E00000000S011',
'V7E00000000S012',
'V7E00000000T001',
'V7E00000000T002',
'V7E00000000T003',
'V7E00000000T004',
'V7E00000000T005',
'V7E00000000T006',
'V7E00000000T007',
'V7E00000000T008',
'V7E00000000T009',
'V7E00000000T010',
'V7E00000000T011',
'V7E00000000T012',
'V7E00000000T013',
'V7E00000000T014',
'V7E00000000T015',
'V7E00000000T016',
'V7E00000000T017',
'V7E00000000T018',
'V7E00000000T019',
'V7E00000000T020',
'V7E00000000T021',
'V7E00000000T022',
'V7E00000000T023',
'V7E00000000T024',
'V7E00000000T025',
'V7E00000000T026',
'V7E00000000T027',
'V7E00000000T028',
'V7E00000000T029',
'V7E00000000T030',
'V7E00000000T031',
'V7E00000000T032',
'V7E00000000T033',
'V7E00000000T034',
'V7E00000000T035',
'V7E00000000T036',
'V7E00000000T037',
'V7E00000000T038',
'V7E00000000T039',
'V7E00000000T040',
'V7E00000000T041',
'V7E00000000T042',
'V7E00000000T043',
'V7E00000000T044',
'V7E00000000T045',
'V7E00000000T046',
'V7E00000000T047',
'V7E00000000S013',
'V7E00000000U001',
'V7E00000000U002',
'V7E00000000U003',
'V7E00000000V001',
'V7E00000000V002',
'V7E00000000V003',
'V7E00000000V004',
'V7E00000000V005',
'V7E00000000V006',
'V7E00000000V007',
'V7E00000000V008',
'V7E00000000W001',
'V7E00000000W002',
'V7E00000000W003',
'V7E00000000W004',
'V7E00000000W005',
'V7E00000000W006',
'V7E00000000V009',
'V7E00000000V010',
'V7E00000000V011',
'V7E00000000W007',
'V7E00000000V012',
'V7E00000000W008',
'V7E00000000W009',
'V7E00000000W010',
'V7E00000000W011',
'V7E00000000V013',
'V7E00000000V014',
'V7E00000000V015',
'V7E00000000V016'
)

In [0]:
%sql
select * from com_edp_prd.wrk_dnli.veeva_crm_export where veevaid__v in ('242976932075930000',
'935167248606169000',
'242976933401330000',
'928924048546203000',
'242977101072827000',
'242976933401330000',
'928924048546203000',
'243241223794459000',
'928924048546203000',
'243159000579834000',
'242988219669414000',
'936459330743962000',
'242977101072827000',
'935167248606169000',
'936459330743962000',
'243235258940523000',
'928924045224052000',
'243148178847695000',
'936459330743962000',
'936459330743962000'
)

In [0]:
%sql
select distinct npi as hco_npi, ORGANIZATION_NAME as hco_name, PROVIDER_ADDRESS as hco_address, PROVIDER_CITY as hco_city, PROVIDER_STATE as hco_state, PROVIDER_ZIP as hco_zip
from com_raw.kom_providers
where PROVIDER_TYPE = 'ORGANIZATION'

In [0]:
%sql
select distinct npi as hco_npi, ORGANIZATION_NAME as hco_name, PROVIDER_ADDRESS as hco_address, PROVIDER_CITY as hco_city, PROVIDER_STATE as hco_state, PROVIDER_ZIP as hco_zip
from com_raw.kom_providers
where PROVIDER_TYPE = 'ORGANIZATION' and npi in (select distinct hco_npi from com_edp_prd.cmpa_insights_internal_schema.reference_file_0109)

In [0]:
%sql 
select * from com_edp_prd.com_raw.kom_providers limit 5;

In [0]:
%sql 
select * from com_edp_prd.com_raw.kom_providers where provider_type = 'ORGANIZATION' and ORGANIZATION_NAME ILIKE '%Driscoll Children%'

In [0]:
%sql
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.patient360 ;  

In [0]:
%sql
with hco_zip_v1 AS (
    SELECT DISTINCT
        a.npi_num__v AS hco_npi_active,
        b.address_line_1__v as hco_address,
        b.postal_code_cda__v AS hco_postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.npi_num__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_hco a
    JOIN com_raw.vod_address b
        ON b.entity_vid__v = a.vid__v
       AND b.entity_type__v = 'HCO'
       AND b.record_state__v = 'VALID'
       AND b.address_status__v IN ('A','DS')
       AND b.address_verification_status__v NOT IN ('NS','U')
    WHERE a.npi_num__v IN (
       '1124796081','1831445014','1548286172'

    )
),
target_hco_npis AS (
    SELECT '1124796081' AS hco_npi_active UNION ALL
    SELECT '1831445014' UNION ALL
    SELECT '1548286172'   
),
 
 
/* ---------------------------------------------------------------------------
   STEP 15: Select latest ZIP per HCO (rn=1)
   --------------------------------------------------------------------------- */
hco_zip_v2 AS (
    SELECT
        hco_npi_active,
        hco_address,
        hco_postal_code
    FROM hco_zip_v1
    WHERE rn = 1
),
 
/* ---------------------------------------------------------------------------
   STEP 16: Enrich / backfill HCO ZIP using:
   1) VOD latest ZIP (preferred)
   2) Komodo provider_zip (fallback)
   --------------------------------------------------------------------------- */
pulling_hco_zip_using_vod_komodo AS (
    SELECT
        a.* ,
        case when b.hco_postal_code is not null then b.hco_address else c.provider_address end as hco_address,
        COALESCE(b.hco_postal_code, c.provider_zip, '-') AS hco_zip
    FROM target_hco_npis a
    LEFT JOIN hco_zip_v2 b
        ON a.hco_npi_active = b.hco_npi_active
    LEFT JOIN com_raw.kom_providers c
        ON a.hco_npi_active = c.npi
       AND c.provider_type = 'ORGANIZATION'
),
 
/* ---------------------------------------------------------------------------
   STEP 17: Reassign territory & region using ZIP
   - Uses HCO ZIP if present; otherwise falls back to HCP ZIP
   --------------------------------------------------------------------------- */
territory_region_reassignment AS (
    SELECT
        a.* ,
        b.territory_id,
        b.territory_name AS territory,
        b.region_id,
        b.region_name AS region,
        city,
        state
    FROM pulling_hco_zip_using_vod_komodo a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON COALESCE(
               TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT),
               TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT)
           ) = b.zipcode
)
 
select a.*, corporate_name__v from territory_region_reassignment as a
left join com_raw.vod_hco on hco_npi_active = npi_num__v

In [0]:
%sql
/* ---------------------------------------------------------------------------
   STEP 1: ZIP input list
   --------------------------------------------------------------------------- */
WITH zip_input AS (
    SELECT '29418' AS zip UNION ALL
    SELECT '92123' UNION ALL
    SELECT '75284'
),

/* ---------------------------------------------------------------------------
   STEP 2: Map ZIP to Territory & Region
   --------------------------------------------------------------------------- */
zip_to_region_territory AS (
    SELECT
        a.zip,
        b.territory_id,
        b.territory_name AS territory,
        b.region_id,
        b.region_name AS region,
        b.city,
        b.state
    FROM zip_input a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON TRY_CAST(a.zip AS BIGINT) = b.zipcode
)

/* ---------------------------------------------------------------------------
   FINAL OUTPUT
   --------------------------------------------------------------------------- */
SELECT *
FROM zip_to_region_territory
ORDER BY region, territory, zip;


In [0]:
%sql
select * from com_edp_prd.com_raw.kom_medical_events where patient_id = 'VM0F6LD8' and DIAGNOSIS_CODES LIKE '%E761%';